# Антибот: поведенческая классификация cookie_id

Python **3.12.14**, seed **42**, один CPU-поток.
Зависимости зафиксированы в `requirements.txt`.


## Подход и защита от утечки

1. Проверяем уникальность куки, метаданные и отсутствие пересечения train/test.
2. Используем **только** события `window_start_ts <= event_ts < window_end_ts`. Удаляем полные дубликаты, сортируем по времени, сохраняем куки без событий.
3. Строим признаки независимо для каждой куки. Метки, сырые идентификаторы объявлений/куки, абсолютные даты и частоты по другим кукам не используются. Нет обучения кодировщиков на test.
4. Baseline: RandomForest по числу событий и уникальных объявлений. Улучшения: ритм и разнообразие поведения, поиск, действия, курсор, нормализованные платформы, общие маркеры UA, возраст куки.
5. Выбираем одну из четырёх заранее заданных конфигураций по среднему официальному P@R≥0.7 на двух последовательных временных разбиениях. При равенстве - по Average Precision.
6. **17–19 апреля. отложенная проверка**, не участвующая в выборе модели. После неё не меняем конфигурацию; обучаем выбранную модель на всём train.

Интервалы событий: среднее, разброс, квантили, доли коротких пауз, повторяемость, сессии с разрывом >30 минут. Разнообразие: число уникальных значений, энтропия, концентрация просмотров. Поиск: глубина, последовательные страницы, длина запросов. Действия: количества и доли типов, отношения к просмотрам объявлений. Курсор: наличие, повторяемость, разброс и длина перемещения между наблюдаемыми точками; это не восстановление непрерывной траектории.

Строки `platform` нормализуются (`WEB`, `desktop` → `web`; `iphone` → `ios`). Полная строка и версии UA исключены для меньшей зависимости от обновлений браузера. Для категорий/локаций/запросов используем разнообразие, а не сами значения. Неопределённые статистики остаются NaN: HGB поддерживает их напрямую, для ExtraTrees imputer обучается только на обучающей части каждого разбиения. Нулём заполняются отсутствующие количества.


In [1]:
import os
for variable in ['OMP_NUM_THREADS', 'OPENBLAS_NUM_THREADS', 'MKL_NUM_THREADS']:
    os.environ[variable] = '1'
from pathlib import Path
DATA_DIR = Path(os.environ.get('ANTIBOT_DATA_DIR', 'data'))
OUTPUT_DIR = Path(os.environ.get('ANTIBOT_OUTPUT_DIR', 'outputs'))
print('Data:', DATA_DIR, 'Outputs:', OUTPUT_DIR)


Data: /workspace/scratch/b31d39b4f69e/upload Outputs: /workspace/scratch/b31d39b4f69e/notebook_verification


## Официальная метрика
Равные score обрабатываются группой. Берётся максимум precision среди **всех** порогов с recall ≥ 0.7.

In [2]:
"""
Куки сортируются по `score` по убыванию. Перебираются все пороги; для каждого
отмечаются куки со `score >= threshold` и считаются precision и recall.
Результат - **максимальный precision среди порогов, у которых recall >= 0.70**.

Одинаковые score обрабатываются **одной группой**. Нельзя разделить куки равны score: либо отмечены все, либо никто. Порядок строк на результат не влияет.

Берётся именно максимум по допустимой области, а не первая точка, где полнота
достигнута. Precision вдоль кривой не монотонен, и эти два числа различаются.
"""

from __future__ import annotations

import numpy as np

TARGET_RECALL = 0.70


def pr_curve(y_true, score) -> tuple[np.ndarray, np.ndarray]:
    """(precision, recall) в точках на границах групп одинакового score."""
    y_true = np.asarray(y_true, dtype=int)
    score = np.asarray(score, dtype=float)
    if y_true.shape != score.shape:
        raise ValueError("y_true и score разной длины")

    n_pos = int(y_true.sum())
    if n_pos == 0:
        return np.array([]), np.array([])

    order = np.argsort(-score, kind="mergesort")
    y, s = y_true[order], score[order]

    tp = np.cumsum(y)
    k = np.arange(1, len(y) + 1)
    ends = np.r_[s[1:] != s[:-1], True]      # последняя строка каждой группы равных score
    return tp[ends] / k[ends], tp[ends] / n_pos


def precision_at_recall(y_true, score, recall: float = TARGET_RECALL) -> float:
    """Максимальный precision среди порогов с recall >= `recall`."""
    prec, rec = pr_curve(y_true, score)
    if len(prec) == 0:
        return float("nan")
    ok = rec >= recall
    return float(prec[ok].max()) if ok.any() else 0.0


def recall_at_fpr(y_true, score, fpr: float = 0.01) -> float:
    """Диагностика: какую долю ботов ловим, задев не более `fpr` доли людей.

    Как и основная метрика, работает по группам одинакового score.
    """
    y_true = np.asarray(y_true, dtype=int)
    score = np.asarray(score, dtype=float)
    n_pos, n_neg = int(y_true.sum()), int((1 - y_true).sum())
    if n_pos == 0 or n_neg == 0:
        return float("nan")

    order = np.argsort(-score, kind="mergesort")
    y, s = y_true[order], score[order]
    ends = np.r_[s[1:] != s[:-1], True]
    tp = np.cumsum(y)[ends]
    fp = np.cumsum(1 - y)[ends]

    allowed = fp <= fpr * n_neg
    return float(tp[allowed].max() / n_pos) if allowed.any() else 0.0




## Загрузка и построение признаков
Все агрегации находятся в одной функции, одинаковой для train и test.

In [3]:
"""Признаки для каждой куки: без целевых меток, глобальных частот и сырых идентификаторов.

Все агрегаты событий вычисляются в интервале [window_start_ts, window_end_ts).
Функция может независимо обрабатывать обучающую, валидационную и тестовую
выборки без обученных кодировщиков.
"""
import numpy as np
import pandas as pd

EVENTS = ['item_view', 'search_results_view', 'photo_swipe', 'favorite_add',
          'seller_page_view', 'contact_phone_show', 'captcha_shown', 'login',
          'contact_chat_open', 'contact_message_sent']


def load_data(path):
    from pathlib import Path
    path = Path(path)
    dates = ['cookie_created_at', 'window_start_ts', 'window_end_ts']
    train = pd.read_csv(path / 'train.csv', parse_dates=dates)
    test = pd.read_csv(path / 'test.csv', parse_dates=dates)
    events = pd.read_csv(path / 'events.csv.gz', parse_dates=['event_ts'])
    for meta in (train, test):
        assert meta.cookie_id.notna().all() and meta.cookie_id.is_unique
        assert meta[dates].notna().all().all()
        assert (meta.window_end_ts > meta.window_start_ts).all()
        assert (meta.cookie_created_at <= meta.window_end_ts).all()
    assert set(train.target.unique()) == {0, 1}
    assert not set(train.cookie_id) & set(test.cookie_id)
    return train, test, events


def build_features(meta, events):
    ev = events.merge(meta[['cookie_id', 'window_start_ts', 'window_end_ts']],
                      on='cookie_id', how='inner', validate='many_to_one')
    inside = ev.event_ts.ge(ev.window_start_ts) & ev.event_ts.lt(ev.window_end_ts)
    outside = int((~inside).sum())
    ev = ev.loc[inside].copy()
    duplicates = int(ev.duplicated().sum())
    ev = ev.drop_duplicates().sort_values(['cookie_id', 'event_ts'], kind='stable')
    ev['platform_norm'] = ev.platform.str.strip().str.lower().replace(
        {'desktop': 'web', 'iphone': 'ios'})
    rows = []
    for cookie, g in ev.groupby('cookie_id', sort=False):
        n = len(g)
        t = g.event_ts.astype('int64').to_numpy() / 1e9
        dt = np.diff(t)
        # Энтропия измеряет концентрацию просмотров. Сырые идентификаторы категорий, запросов,
        # местоположений и объявлений намеренно не подаются на вход модели.
        f = {'cookie_id': cookie, 'n_events': n,
             'active_span_s': t[-1] - t[0],
             'first_offset_s': (g.event_ts.iloc[0] - g.window_start_ts.iloc[0]).total_seconds(),
             'last_to_end_s': (g.window_end_ts.iloc[0] - g.event_ts.iloc[-1]).total_seconds(),
             'active_hours': g.event_ts.dt.hour.nunique(),
             'active_minutes': g.event_ts.dt.floor('min').nunique(),
             'night_fraction': g.event_ts.dt.hour.between(0, 5).mean(),
             'sessions_30min': 1 + np.sum(dt > 1800),
             'same_second_fraction': 1 - g.event_ts.nunique() / n}
        f['events_per_active_minute'] = n / max(f['active_span_s'] / 60, 1)
        for col in ['item_id', 'item_category', 'item_location', 'seller_type',
                    'search_query', 'user_agent', 'platform_norm', 'event_name']:
            counts = g[col].value_counts()
            p = counts.to_numpy() / max(counts.sum(), 1)
            f[col + '_nunique'] = len(counts)
            f[col + '_missing'] = g[col].isna().mean()
            f[col + '_entropy'] = -np.sum(p * np.log2(p)) if len(p) else np.nan
            f[col + '_top_fraction'] = p.max() if len(p) else np.nan
            f[col + '_unique_ratio'] = len(counts) / max(g[col].notna().sum(), 1)
        vc = g.event_name.value_counts()
        for name in EVENTS:
            f['count_' + name] = vc.get(name, 0)
            f['fraction_' + name] = vc.get(name, 0) / n
        for platform in ['web', 'android', 'ios']:
            f['platform_' + platform] = g.platform_norm.eq(platform).mean()
        for name, pattern in {
            'headless': 'headless', 'automation': 'python|requests|curl|wget|httpclient|scrapy|aiohttp',
            'chrome': 'chrome', 'firefox': 'firefox', 'safari': 'safari',
            'yandex': 'yabrowser', 'mobile': 'mobile|android|iphone',
            'linux': 'linux', 'windows': 'windows', 'mac': 'macintosh',
        }.items():
            f['ua_' + name] = g.user_agent.str.contains(pattern, case=False, regex=True, na=False).mean()
        # Номера версий и исправлений браузера, а также полные строки User-Agent исключены.
        if len(dt):
            for key, value in [('mean', dt.mean()), ('std', dt.std()), ('min', dt.min()),
                               ('max', dt.max()), ('cv', dt.std() / max(dt.mean(), .001))]:
                f['gap_' + key] = value
            for q in [.1, .25, .5, .75, .9]:
                f['gap_q' + str(int(q * 100))] = np.quantile(dt, q)
            for threshold in [1, 2, 5, 10, 60, 300, 1800]:
                f['gap_le_' + str(threshold)] = np.mean(dt <= threshold)
            f['gap_unique_ratio'] = len(np.unique(dt)) / len(dt)
            f['gap_top_fraction'] = pd.Series(dt).value_counts(normalize=True).iloc[0]
            f['same_event_adjacent'] = np.mean(g.event_name.to_numpy()[1:] == g.event_name.to_numpy()[:-1])
            f['event_transition_nunique'] = len(set(zip(g.event_name.iloc[:-1], g.event_name.iloc[1:])))
        bins = g.event_ts.dt.floor('min').value_counts()
        f['max_events_minute'] = bins.max()
        f['minute_top_fraction'] = bins.max() / n
        f['hour_top_fraction'] = g.event_ts.dt.hour.value_counts(normalize=True).max()
        search = g.loc[g.event_name.eq('search_results_view')]
        pages = search.search_page.dropna().to_numpy()
        if len(pages):
            f['page_mean'] = pages.mean()
            f['page_max'] = pages.max()
            f['page_std'] = pages.std()
            f['page_unique'] = len(np.unique(pages))
            f['page_gt_3'] = np.mean(pages > 3)
            f['page_gt_10'] = np.mean(pages > 10)
            if len(pages) > 1:
                f['page_next_fraction'] = np.mean(np.diff(pages) == 1)
                f['page_nondec_fraction'] = np.mean(np.diff(pages) >= 0)
        queries = search.search_query.dropna().astype(str)
        f['query_len_mean'] = queries.str.len().mean()
        pointer = g[['pointer_x', 'pointer_y']].dropna()
        f['pointer_present'] = len(pointer) / n
        web_n = g.platform_norm.eq('web').sum()
        f['pointer_per_web_event'] = len(pointer) / web_n if web_n else np.nan
        if len(pointer):
            f['pointer_unique_ratio'] = len(pointer.drop_duplicates()) / len(pointer)
            for col in pointer:
                f[col + '_std'] = pointer[col].std()
                f[col + '_range'] = pointer[col].max() - pointer[col].min()
            if len(pointer) > 1:
                dist = np.linalg.norm(np.diff(pointer.to_numpy(), axis=0), axis=1)
                f['pointer_step_mean'] = dist.mean()
                f['pointer_step_std'] = dist.std()
                f['pointer_zero_step'] = np.mean(dist == 0)
        views = vc.get('item_view', 0)
        for name in ['photo_swipe', 'favorite_add', 'seller_page_view', 'contact_phone_show',
                     'contact_chat_open', 'contact_message_sent', 'search_results_view']:
            f[name + '_per_view'] = vc.get(name, 0) / max(views, 1)
        rows.append(f)
    f = pd.DataFrame(rows).set_index('cookie_id').reindex(meta.cookie_id)
    zero_cols = [c for c in f if c.startswith(('count_', 'fraction_', 'platform_', 'ua_'))
                 or c.endswith('_nunique')] + ['n_events']
    f[zero_cols] = f[zero_cols].fillna(0)
    f['cookie_age_days'] = ((meta.window_end_ts - meta.cookie_created_at).dt.total_seconds() / 86400).to_numpy()
    f['age_at_start_days'] = ((meta.window_start_ts - meta.cookie_created_at).dt.total_seconds() / 86400).to_numpy()
    f['created_in_window'] = meta.cookie_created_at.ge(meta.window_start_ts).astype(int).to_numpy()
    f['window_duration_hours'] = ((meta.window_end_ts - meta.window_start_ts).dt.total_seconds() / 3600).to_numpy()
    f = f.replace([np.inf, -np.inf], np.nan).astype(float)
    audit = {'cookies': len(meta), 'in_window_deduplicated_events': len(ev),
             'outside_window_events': outside, 'duplicate_events_removed': duplicates,
             'cookies_without_events': int(f.n_events.eq(0).sum()), 'features': len(f.columns)}
    return f, audit


## Обучение и временная валидация
Внутри обучения нет случайной валидации для early stopping: число итераций фиксировано. Балансировку классов и подбор порога не используем - проверяющая система сама перебирает пороги. Интервал bootstrap - описательная оценка по кукам, не гарантия качества на новых источниках.

In [4]:
"""
Вычисления только на CPU, без сетевых запросов, внешних моделей, ручной разметки
и эвристик по идентификаторам. Модель выбирается на двух последовательных
временных валидационных блоках. Последние три даты обучающей выборки
отложены до завершения выбора. Затем модель переобучается на всей обучающей выборке.
"""
import os
for variable in ['OMP_NUM_THREADS', 'OPENBLAS_NUM_THREADS', 'MKL_NUM_THREADS']:
    os.environ[variable] = '1'

import argparse
import hashlib
import json
import platform
from pathlib import Path
import numpy as np
import pandas as pd
import sklearn
from sklearn.ensemble import HistGradientBoostingClassifier, ExtraTreesClassifier, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.metrics import average_precision_score, roc_auc_score

SEED = 42
BASIC = ['n_events', 'item_id_nunique']
CANDIDATES = ['baseline_rf', 'hgb_15', 'hgb_31', 'extra_trees']


def make_model(name):
    if name == 'baseline_rf':
        return RandomForestClassifier(n_estimators=300, min_samples_leaf=3,
                                      random_state=SEED, n_jobs=1)
    if name == 'extra_trees':
        return make_pipeline(SimpleImputer(strategy='median', add_indicator=True, keep_empty_features=True),
                             ExtraTreesClassifier(n_estimators=400, min_samples_leaf=2,
                                                  max_features=.8, random_state=SEED, n_jobs=1))
    return HistGradientBoostingClassifier(
        max_iter=300, learning_rate=.07, max_leaf_nodes=15 if name == 'hgb_15' else 31,
        min_samples_leaf=25, l2_regularization=10, early_stopping=False, random_state=SEED)


def metrics(y, p):
    return {'precision_at_recall_70': precision_at_recall(y, p),
            'average_precision': average_precision_score(y, p),
            'roc_auc': roc_auc_score(y, p)}


def run(data_dir='data', output_dir='outputs'):
    out = Path(output_dir)
    out.mkdir(parents=True, exist_ok=True)
    train, test, events = load_data(data_dir)
    print('Building train features...', flush=True)
    x, audit_train = build_features(train, events)
    print('Building test features...', flush=True)
    xt, audit_test = build_features(test, events)
    xt = xt.reindex(columns=x.columns)
    print('Data audit:', audit_train, audit_test, flush=True)
    y = train.target.to_numpy()
    dates = sorted(train.window_start_ts.unique())
    assert len(dates) >= 10, 'At least ten distinct observation dates are required.'
    # Границы разбиений зависят только от дат, а не от целевых меток или результатов моделей.
    splits = [(dates[-8], dates[-5]), (dates[-5], dates[-3])]
    rows = []
    for name in CANDIDATES:
        cols = BASIC if name == 'baseline_rf' else list(x.columns)
        for k, (start, end) in enumerate(splits):
            fit = train.window_start_ts.lt(start).to_numpy()
            valid = (train.window_start_ts.ge(start) & train.window_start_ts.lt(end)).to_numpy()
            model = make_model(name)
            model.fit(x.loc[fit, cols], y[fit])
            p = model.predict_proba(x.loc[valid, cols])[:, 1]
            row = {'model': name, 'fold': k + 1, 'train_n': int(fit.sum()),
                   'valid_n': int(valid.sum()), 'valid_positive': int(y[valid].sum()),
                   'valid_start': str(pd.Timestamp(start).date()),
                   'valid_end_exclusive': str(pd.Timestamp(end).date()), **metrics(y[valid], p)}
            rows.append(row)
            print(row, flush=True)
    results = pd.DataFrame(rows)
    results.to_csv(out / 'experiments.csv', index=False)
    ranked = results.groupby('model')[['precision_at_recall_70', 'average_precision']].mean()
    selected = ranked.sort_values(['precision_at_recall_70', 'average_precision'], ascending=False).index[0]
    print('Selected using development folds only:', selected, flush=True)
    fit = train.window_start_ts.lt(dates[-3]).to_numpy()
    holdout = ~fit
    holdout_results = {}
    predictions = pd.DataFrame({'cookie_id': train.loc[holdout, 'cookie_id'],
                                'date': train.loc[holdout, 'window_start_ts'], 'target': y[holdout]})
    for name in dict.fromkeys(['baseline_rf', selected]):
        cols = BASIC if name == 'baseline_rf' else list(x.columns)
        model = make_model(name).fit(x.loc[fit, cols], y[fit])
        p = model.predict_proba(x.loc[holdout, cols])[:, 1]
        predictions[name] = p
        holdout_results[name] = metrics(y[holdout], p)
    predictions.to_csv(out / 'holdout_predictions.csv', index=False)
    daily = []
    for date, group in predictions.groupby('date'):
        daily.append({'date': str(date.date()), 'n': len(group), 'positive': int(group.target.sum()),
                      **metrics(group.target, group[selected])})
    # Только оценка неопределённости: по результатам этой отложенной выборки модель не меняем.
    rng = np.random.default_rng(SEED)
    yy, pp = y[holdout], predictions[selected].to_numpy()
    boot = []
    for _ in range(500):
        idx = rng.integers(0, len(yy), len(yy))
        boot.append(precision_at_recall(yy[idx], pp[idx]))
    report = {
        'seed': SEED, 'python': platform.python_version(), 'numpy': np.__version__,
        'pandas': pd.__version__, 'scikit_learn': sklearn.__version__,
        'selected_model': selected, 'train_positive': int(y.sum()),
        'train_audit': audit_train, 'test_audit': audit_test,
        'selection_mean_metrics': ranked.to_dict(orient='index'),
        'holdout_start': str(pd.Timestamp(dates[-3]).date()),
        'holdout_n': int(holdout.sum()), 'holdout_positive': int(yy.sum()),
        'holdout_metrics': holdout_results, 'holdout_daily': daily,
        'holdout_cookie_bootstrap_95_percentile': np.quantile(boot, [.025, .975]).tolist(),
        'bootstrap_caveat': 'Cookie resampling ignores dependence between traffic sources and dates; descriptive only.',
        'input_sha256': {name: hashlib.sha256((Path(data_dir) / name).read_bytes()).hexdigest()
                         for name in ['train.csv', 'test.csv', 'events.csv.gz']},
    }
    cols = BASIC if selected == 'baseline_rf' else list(x.columns)
    final = make_model(selected).fit(x[cols], y)
    sub = pd.DataFrame({'cookie_id': test.cookie_id.to_numpy(),
                        'score': final.predict_proba(xt[cols])[:, 1]})
    assert sub.columns.tolist() == ['cookie_id', 'score']
    assert sub.cookie_id.is_unique and sub.cookie_id.equals(test.cookie_id)
    assert len(sub) == len(test) and sub.score.notna().all()
    assert np.isfinite(sub.score).all() and sub.score.between(0, 1).all()
    sub.to_csv(out / 'submission.csv', index=False, float_format='%.17g')
    # Проверяем и сохранённый файл, и таблицу данных в памяти.
    check = pd.read_csv(out / 'submission.csv')
    assert check.cookie_id.equals(test.cookie_id) and check.score.between(0, 1).all()
    report['submission_sha256'] = hashlib.sha256((out / 'submission.csv').read_bytes()).hexdigest()
    (out / 'report.json').write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding='utf-8')
    (out / 'feature_names.txt').write_text('\n'.join(cols), encoding='utf-8')
    print(json.dumps(report, ensure_ascii=False, indent=2), flush=True)
    return report




## Запуск экспериментов и создание submission
Результаты сохраняются вместе с версиями библиотек, seed, контрольными суммами исходных файлов и submission.

In [5]:
report = run(DATA_DIR, OUTPUT_DIR)

Building train features...
Building test features...
Data audit: {'cookies': 11091, 'in_window_deduplicated_events': 195484, 'outside_window_events': 40779, 'duplicate_events_removed': 2952, 'cookies_without_events': 0, 'features': 137} {'cookies': 4909, 'in_window_deduplicated_events': 88349, 'outside_window_events': 0, 'duplicate_events_removed': 1341, 'cookies_without_events': 0, 'features': 137}
{'model': 'baseline_rf', 'fold': 1, 'train_n': 5113, 'valid_n': 2486, 'valid_positive': 190, 'valid_start': '2026-04-12', 'valid_end_exclusive': '2026-04-15', 'precision_at_recall_70': 0.08866995073891626, 'average_precision': 0.2425108179641357, 'roc_auc': 0.6170089858793325}
{'model': 'baseline_rf', 'fold': 2, 'train_n': 7599, 'valid_n': 1541, 'valid_positive': 125, 'valid_start': '2026-04-15', 'valid_end_exclusive': '2026-04-17', 'precision_at_recall_70': 0.10388127853881278, 'average_precision': 0.20813842572589053, 'roc_auc': 0.6264406779661017}
{'model': 'hgb_15', 'fold': 1, 'train_n'

## Проверка результата и сравнение
Последний holdout не использован для подбора конфигурации. Публикуются результаты обоих development-разбиений, а не только лучший эксперимент.

In [6]:
print(pd.read_csv(OUTPUT_DIR / 'experiments.csv').to_string(index=False))
print('\nHoldout metrics:')
print(pd.DataFrame(report['holdout_metrics']).T.to_string())
submission = pd.read_csv(OUTPUT_DIR / 'submission.csv')
print('\nSubmission:', submission.shape)
print(submission.head().to_string(index=False))
print('SHA256:', report['submission_sha256'])


      model  fold  train_n  valid_n  valid_positive valid_start valid_end_exclusive  precision_at_recall_70  average_precision  roc_auc
baseline_rf     1     5113     2486             190  2026-04-12          2026-04-15                0.088670           0.242511 0.617009
baseline_rf     2     7599     1541             125  2026-04-15          2026-04-17                0.103881           0.208138 0.626441
     hgb_15     1     5113     2486             190  2026-04-12          2026-04-15                0.700000           0.763192 0.918985
     hgb_15     2     7599     1541             125  2026-04-15          2026-04-17                0.684211           0.770685 0.935808
     hgb_31     1     5113     2486             190  2026-04-12          2026-04-15                0.711230           0.756381 0.913325
     hgb_31     2     7599     1541             125  2026-04-15          2026-04-17                0.692913           0.767992 0.937412
extra_trees     1     5113     2486             

## Ограничения и интерпретация

- Нет идентификатора сервиса/источника разметки: нельзя провести leave-one-service-out и доказать обобщение на ранее неизвестных ботов. Временная проверка проверяет лишь доступный сдвиг дат.
- Всего 14 дней train; test идёт позже. Ни локальные метрики, ни bootstrap не гарантируют качество на скрытом test. Максимизация по порогам внутри метрики может давать оптимистичную оценку при небольшом числе положительных примеров.
- Показ капчи - признак работы существующей защиты. Он доступен до конца окна, но при изменении политики капчи зависимость может ухудшить переносимость. Аналогично, наличие координат зависит от платформы и инструментирования.
- Полные дубликаты удаляются как повторы логирования. Без уникального ID события нельзя исключить, что часть полных совпадений - реальные повторные действия в ту же секунду.
- Суточные события нельзя использовать для блокировки в начале дня: решение рассчитано на оценку **после окончания окна**.
- Raw score не калибровался; это ранжирующая оценка в [0,1]. Для эксплуатационного порога нужна отдельная последующая валидация.
- Метки отражают известных сборщиков и трафик, отнесённый к человеческому, а не идеально полную разметку всех видов автоматизации.
